# Speculative decoding benchmark (HF)

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q transformers accelerate bitsandbytes pandas matplotlib

## Config

In [ ]:
TARGET_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
TARGET_QUANTIZATION = None

DRAFT_CONFIGS = [
    {"model_path": None, "quantization": None, "label": "baseline"},
    {"model_path": "Qwen/Qwen2.5-1.5B-Instruct", "quantization": "4bit", "label": "self-4bit"},
    {"model_path": "Qwen/Qwen2.5-0.5B-Instruct", "quantization": None, "label": "0.5B-fp16"},
    {"model_path": "Qwen/Qwen2.5-0.5B-Instruct", "quantization": "4bit", "label": "0.5B-4bit"},
]

NUM_ASSISTANT_TOKENS_LIST = [0, 3, 5, 8]
MAX_NEW_TOKENS = 256
NUM_PROMPTS = 8
OUTPUT_FILE = "bench_results.jsonl"

In [ ]:
from bench_hf import load_model, load_tokenizer, get_gpu_memory_info, get_model_memory_mb

target_model = load_model(TARGET_MODEL, quantization=TARGET_QUANTIZATION)
tokenizer = load_tokenizer(TARGET_MODEL)

print(f"{get_model_memory_mb(target_model):.0f} MB, {target_model.dtype}, {target_model.device}")
print(get_gpu_memory_info())

## Run

In [ ]:
import os
from bench_hf import run_multi_draft_benchmark

if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)

results = run_multi_draft_benchmark(
    target_model, tokenizer,
    draft_configs=DRAFT_CONFIGS,
    num_assistant_tokens_list=NUM_ASSISTANT_TOKENS_LIST,
    max_new_tokens=MAX_NEW_TOKENS,
    num_prompts=NUM_PROMPTS,
    output_file=OUTPUT_FILE,
)

## Results



In [ ]:
import pandas as pd

df = pd.DataFrame(results)
display(df[["draft_label", "num_assistant_tokens", "speed_tok_s", "acc_length", "step_time_ms", "total_time_s"]])

In [ ]:
import matplotlib.pyplot as plt

df["label"] = df["draft_label"] + "\nnat=" + df["num_assistant_tokens"].astype(str)
colors = ["steelblue" if x else "seagreen" for x in df["is_baseline"]]
xs = range(len(df))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col, title in zip(axes, ["speed_tok_s", "acc_length", "step_time_ms"], ["speed tok/s", "acc length", "step ms"]):
    ax.bar(xs, df[col], color=colors)
    ax.set_xticks(xs)
    ax.set_xticklabels(df["label"], rotation=45, ha="right", fontsize=7)
    ax.set_title(title)

plt.tight_layout()
plt.savefig("bench_results.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
baseline_speed = df[df["is_baseline"]]["speed_tok_s"].mean()
spec_rows = df[~df["is_baseline"]].copy()
spec_rows["speedup"] = spec_rows["speed_tok_s"] / baseline_speed

display(spec_rows[["draft_label", "num_assistant_tokens", "speed_tok_s", "acc_length", "speedup"]])

best = spec_rows.loc[spec_rows["speedup"].idxmax()]
print(f"best: {best['draft_label']} nat={int(best['num_assistant_tokens'])} -> {best['speedup']:.2f}x")

In [ ]:
import numpy as np

pivot = spec_rows.pivot_table(index="draft_label", columns="num_assistant_tokens", values="speedup")

fig, ax = plt.subplots(figsize=(8, 3))
im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=0.5)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f"{pivot.values[i, j]:.2f}x", ha="center", va="center")
plt.colorbar(im)
plt.tight_layout()
plt.savefig("speedup_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()

## Reload results (если ядро упало)

In [ ]:
# import json, pandas as pd
# results = [json.loads(x) for x in open("bench_results.jsonl")]
# df = pd.DataFrame(results)